# Example 11.6 / 4: Complementarity Is Not Multiplication

Companion notebook for **11.6 Complementarity Constraints** in the AMPL lab Documentation, section "Complementarity Is Not Multiplication". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Complementarity Is Not Multiplication

> **Warning**
> `complements` is not multiplication. Replacing it with `x * F(x) = 0` creates a nonlinear equality and can give a solver a materially different and harder representation.

The market pair written as a product, `Price * (2*Price - 6) = 0` with the two inequalities kept,
is a nonlinear equality (`_snnlcons = 1`). Ipopt started from `Price = 1` reports local
infeasibility, while the `complements` form above is solved at `Price = 3`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var Price >= 0, := 1;  # market price, started at 1

subject to Product: Price * (2 * Price - 6) = 0;  # complementarity as a product: nonlinear
subject to Supply: 2 * Price >= 6;                # excess supply nonnegative

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, round(ampl.var["Price"].value(), 6), ampl.get_value("_snnlcons"))

infeasible 0.0 1
